# 🎾 Padel Analytics — Broadcast-Grade Player Tracking

> **Turn a raw padel clip into a clean, AR-style tracking reel** — the kind of overlay you see on
> a TV sports broadcast: each player wrapped in a ground ellipse, a colored motion trail, a
> persistent ID, and a live info panel.

This notebook is the visual flagship of the project. We go from a noisy frame (the detector sees
*everyone* — players **and** the crowd) to a polished, four-player reel, one honest step at a time:

1. **Peek** at a raw 1080p frame
2. **Detect** every person with **RF-DETR** (Roboflow's transformer detector — no NMS)
3. **Isolate the court** to keep only the 4 on-court players
4. **Track** them with **ByteTrack** so each keeps a stable identity
5. **Design** the broadcast / AR look (ground ellipses, pointers, motion trails, info panel)
6. **Render** the hero reel (MP4 + a lightweight GIF for sharing)
7. **Analytics** — turn the tracks into player movement trajectories

**Stack:** RF-DETR · ByteTrack · Supervision · OpenCV · NumPy


## ⚙️ Setup — run this first

This notebook is **self-contained** — it needs no local project files, so you can open it on
**Google Colab**, upload your own padel clip, and run it top to bottom. This first cell installs the
dependencies (a no-op if they're already present locally).

In [ ]:
import importlib.util, subprocess, sys

# cv2 / numpy / matplotlib / pandas / tqdm / torch ship with Colab; install the rest if missing.
for pkg, mod in [("rfdetr", "rfdetr"), ("ultralytics", "ultralytics"),
                 ("supervision", "supervision"), ("ipywidgets", "ipywidgets")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
print("dependencies ready")

## 0. Setup

Imports, paths, a colour palette, and a tiny `show()` helper for inline frames.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keep the narrative output clean

from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import supervision as sv
from tqdm.auto import tqdm

from supervision.annotators.utils import ColorLookup

# --- self-contained stand-ins for the project package (no local src/ required) ---
import shutil, subprocess, torch


class _DetectorCfg:
    """Minimal stand-in for padel_analytics' Config().detector."""
    model = "medium"        # rfdetr size: nano | small | medium | base | large
    confidence = 0.5
    device = None           # None -> cuda if available, else cpu


class Config:
    detector = _DetectorCfg()


class _RFDETRDetector:
    """RF-DETR person detector -> canonical 'player' sv.Detections."""

    def __init__(self, cfg):
        import rfdetr
        table = {"nano": "RFDETRNano", "small": "RFDETRSmall", "medium": "RFDETRMedium",
                 "base": "RFDETRBase", "large": "RFDETRLarge"}
        name = table.get(getattr(cfg, "model", "medium"), "RFDETRBase")
        name = name if hasattr(rfdetr, name) else "RFDETRBase"
        self.device = getattr(cfg, "device", None) or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = getattr(rfdetr, name)(device=self.device)
        self.conf = getattr(cfg, "confidence", 0.5)

    def detect(self, frame_bgr):
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        det = self.model.predict(rgb, threshold=self.conf)
        det = det[det.class_id == 1]                      # COCO person
        det.class_id = np.zeros(len(det), dtype=int)      # canonical "player"
        return det


def build_detector(cfg):
    return _RFDETRDetector(cfg)


def to_h264(source, target=None):
    """Transcode to browser-friendly H.264 (returns the source unchanged if ffmpeg is missing)."""
    source = Path(source)
    if shutil.which("ffmpeg") is None:
        return source
    target = Path(target) if target else source.with_name(f"{source.stem}_h264.mp4")
    subprocess.run(["ffmpeg", "-y", "-i", str(source), "-c:v", "libx264", "-pix_fmt", "yuv420p",
                    "-movflags", "+faststart", "-an", str(target)],
                   check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return target

plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (13, 7)

# Your clip: use the local sample if present, otherwise upload one (e.g. on Google Colab).
VIDEO = Path("../../data/raw/padel_clip.mp4")
if not VIDEO.exists():
    try:
        from google.colab import files
        print("Upload a short padel clip (.mp4):")
        VIDEO = Path(next(iter(files.upload())))
    except Exception as e:
        raise FileNotFoundError("Set VIDEO to your clip path (or run on Colab to upload one).") from e

OUT = Path("../../data/processed") if Path("../../data").exists() else Path("outputs")
OUT.mkdir(parents=True, exist_ok=True)

info = sv.VideoInfo.from_video_path(str(VIDEO))
FPS = int(round(info.fps))
print(f"{VIDEO.name}: {info.width}x{info.height} @ {info.fps:g} fps, {info.total_frames} frames")


def grab(idx: int):
    '''Random-access a single frame by index (fast seek).'''
    cap = cv2.VideoCapture(str(VIDEO))
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, f = cap.read(); cap.release()
    if not ok:
        raise IndexError(idx)
    return f


def show(img_bgr, title=None, ax=None):
    ax = ax or plt.gca()
    ax.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=13, weight="bold")


## 1. A single raw frame

An elevated end-court view of a padel match — four players on court, spectators in the stands, and a couple of other courts visible in the background.

In [ ]:
frame = grab(300)  # ~5 seconds in
plt.figure(); show(frame, "Raw frame (1920×1080)"); plt.show()

## 2. Detect people with RF-DETR

We use **RF-DETR** (a DETR-style transformer detector) through the project's swappable detector
interface. It's COCO-pretrained, so it already knows the `person` class — no training required.

Notice the problem: the model faithfully detects **everyone**, including the crowd. For a clean
analytics overlay we only want the four players on court.

In [ ]:
detector = build_detector(Config().detector)   # RF-DETR medium, GPU if available
raw = detector.detect(frame)
print(f"RF-DETR found {len(raw)} people in this frame")

box = sv.RoundBoxAnnotator(thickness=2)
plt.figure()
show(box.annotate(frame.copy(), raw), f"RF-DETR detects everyone — {len(raw)} people (players + crowd)")
plt.show()

## 3. Isolate the court

The four players all stand on the blue court; almost every false positive is a spectator outside it.
So we define a **court polygon** and keep only detections whose feet (bottom-centre) fall inside it,
using Supervision's `PolygonZone`.

> This polygon is hand-placed for this clip. **Stage 3** of the roadmap replaces it with automatic
> court-line detection + homography — but a fixed zone is all we need to get a clean reel today.

In [ ]:
# Court polygon (x, y) in full-resolution pixels — tweak the points and re-run to fit your clip.
COURT = np.array([(640, 215), (1270, 215), (1480, 470),
                  (1520, 1060), (380, 1060), (430, 470)], dtype=np.int32)
zone = sv.PolygonZone(polygon=COURT)


def draw_court(img, color=(255, 255, 255), alpha=0.65, thickness=3):
    ov = img.copy()
    cv2.polylines(ov, [COURT], True, color, thickness, cv2.LINE_AA)
    return cv2.addWeighted(ov, alpha, img, 1 - alpha, 0)


players = raw[zone.trigger(raw)]
print(f"{len(raw)} detections  ->  {len(players)} inside the court")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
show(draw_court(box.annotate(frame.copy(), raw)), "Everyone + court zone", axes[0])
show(draw_court(box.annotate(frame.copy(), players)), f"Filtered: {len(players)} on-court players", axes[1])
plt.tight_layout(); plt.show()

## 4. Give each player a stable identity

Detection is per-frame and anonymous. To track a player across time we feed the filtered detections
to **ByteTrack**, which links them frame-to-frame and assigns a persistent `tracker_id`. We give it a
~1-second memory so a player isn't lost behind the net for a moment.

In [ ]:
def new_tracker():
    return sv.ByteTrack(frame_rate=FPS, lost_track_buffer=FPS)  # ~1s of occlusion tolerance

tracked = new_tracker().update_with_detections(players)
print("Player IDs this frame:", sorted(int(t) for t in tracked.tracker_id))

## 5. Make it Augmented Reality — a projected ground ring

A bounding box screams *"computer-vision demo"*. Broadcast AR graphics (think the glowing rings under
players on a football telecast) feel like they're **painted on the pitch** because they're a real
circle **projected through the camera onto the ground plane** — not a flat sticker.

So instead of guessing an ellipse, we build the ring properly: take a circle of constant radius, place
it flat on the ground, then **rotate it in 3D (tilt / yaw / roll) and apply perspective** before
drawing. A few intuitive knobs shape it:

- **tilt (X)** — lays the circle down onto the floor (the main flatten)
- **rotate Y / rotate Z** — yaw & in-plane spin to line the ellipse up with the court's perspective
- **perspective** — how strongly the far side foreshortens (the broadcast "keystone")
- **radius** — one constant size for every player, just like the real thing
- **depth** — a touch of shrink-with-distance so far players don't get oversized rings

The result is thin, clean and anti-aliased — and in the next cell you can **dial it in live** with
sliders.

In [ ]:
PALETTE = sv.ColorPalette.from_hex(["#00E5FF", "#FF3DAE", "#FFD23F", "#7CFF6B"])
LK = ColorLookup.TRACK
FRAME_H = info.height

# One shared parameter set for the ground ring — tweak live with the sliders below.
RING = dict(radius=70, tilt=68.0, rot_y=0.0, rot_z=0.0, persp=420.0, depth=0.45,
            thickness=2, glow=0.55)


def _rot(axis, deg):
    a = np.radians(deg); c, s = np.cos(a), np.sin(a)
    if axis == "x": return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])
    if axis == "y": return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def ring_points(center, p, n=160):
    '''Project a constant-radius ground circle through tilt/yaw/roll + perspective.'''
    cx, cy = center
    scale = 1.0 + p["depth"] * ((cy / FRAME_H) - 0.5) * 2.0       # gentle shrink with distance
    r = p["radius"] * max(0.25, scale)
    th = np.linspace(0, 2 * np.pi, n, endpoint=False)
    pts = np.stack([np.cos(th), np.sin(th), np.zeros_like(th)], 1) * r   # circle in the ground plane
    R = _rot("x", p["tilt"]) @ _rot("y", p["rot_y"]) @ _rot("z", p["rot_z"])
    pts = pts @ R.T
    f = p["persp"]; factor = f / np.clip(f - pts[:, 2], 1e-3, None)      # perspective divide
    return np.stack([cx + pts[:, 0] * factor, cy + pts[:, 1] * factor], 1)


def draw_ring(img, pts, color, thickness=2, glow=0.1):
    '''Thin, broadcast-clean double ring with a soft glow.'''
    pi = pts.astype(np.int32)
    if glow > 0:
        layer = np.zeros_like(img)
        cv2.polylines(layer, [pi], True, color, thickness + 7, cv2.LINE_AA)
        cv2.polylines(layer, [pi], True, color, 7, cv2.LINE_AA)     # dark base for contrast
        cv2.polylines(layer, [pi], True, color, 2, cv2.LINE_AA)            # colour rim

        img = cv2.addWeighted(img, 1.0, cv2.GaussianBlur(layer, (0, 0), 7), glow, 0)
    inner = ((pts - pts.mean(0)) * 0.80 + pts.mean(0)).astype(np.int32)
    cv2.polylines(img, [inner], True, (255, 255, 255), 1, cv2.LINE_AA)       # white inner highlight
    return img


def ground_rings(img, dets, p=RING):
    '''Projected AR ground ring for every tracked player, coloured by track ID.'''
    if dets.tracker_id is None:
        return img
    for xyxy, tid in zip(dets.xyxy, dets.tracker_id):
        x1, y1, x2, y2 = xyxy
        color = PALETTE.by_idx(int(tid)).as_bgr()
        pts = ring_points(((x1 + x2) / 2, float(y2)), p)
        img = draw_ring(img, pts, color, p["thickness"], p["glow"])
    return img


plt.figure()
show(ground_rings(draw_court(frame.copy(), alpha=0.30, thickness=2), tracked),
     "Projected AR ground rings")
plt.show()

### 🎛️ Dial it in — interactive controls

Run the cell below and drag the sliders to shape the rings live on this frame. The values write back
to the shared `RING` dict, so whatever look you settle on is exactly what the reel render (Step 6)
uses. *(Sliders need a live kernel — `jupyter lab`/`notebook`; on a static view you'll just see the
preview at the current settings.)*

In [ ]:
import ipywidgets as widgets

def _preview(**kw):
    RING.update(kw)
    plt.figure(figsize=(12, 7))
    show(ground_rings(draw_court(frame.copy(), alpha=0.30, thickness=2), tracked),
         "Tune the AR rings — live preview")
    plt.show()

def _sl(cls, value, lo, hi, step, desc):
    return cls(value=value, min=lo, max=hi, step=step, description=desc,
              continuous_update=False, readout_format=".0f" if cls is widgets.IntSlider else ".2f")

widgets.interact(
    _preview,
    radius=_sl(widgets.IntSlider, RING["radius"], 20, 160, 2, "radius"),
    tilt=_sl(widgets.FloatSlider, RING["tilt"], 0, 89, 1, "tilt (X)"),
    rot_y=_sl(widgets.FloatSlider, RING["rot_y"], -45, 45, 1, "rotate Y"),
    rot_z=_sl(widgets.FloatSlider, RING["rot_z"], -180, 180, 1, "rotate Z"),
    persp=_sl(widgets.FloatSlider, RING["persp"], 120, 2000, 20, "perspective"),
    depth=_sl(widgets.FloatSlider, RING["depth"], 0.0, 1.2, 0.05, "depth"),
    thickness=_sl(widgets.IntSlider, RING["thickness"], 1, 6, 1, "thickness"),
    glow=_sl(widgets.FloatSlider, RING["glow"], 0.0, 1.0, 0.05, "glow"),
);

### Bring the players to the front

Right now the overlays are painted *over* everyone. To make it feel real, the players should sit **in
front** of the ground rings and the heatmap. We grab a quick **segmentation matte** of the on-court
players (YOLO11-seg) and re-paste their original pixels on top of the overlay — so the ring passes
*behind* their legs and they pop out of the heatmap. It's purely additive: detection, tracking, rings
and the heatmap stay exactly as they are.

In [ ]:
from ultralytics import YOLO

_SEG = YOLO("yolo11n-seg.pt")             # person segmentation for the foreground matte
FOREGROUND = True                          # set False to disable players-in-front compositing


def player_matte(img, erode=2, feather=1.0):
    """Soft mask of the on-court players (persons whose feet fall inside COURT)."""
    res = _SEG(img, verbose=False, classes=[0], retina_masks=True)[0]
    m = np.zeros(img.shape[:2], np.float32)
    if res.masks is not None:
        for poly, box in zip(res.masks.xy, res.boxes.xyxy.cpu().numpy()):
            x1, y1, x2, y2 = box
            if cv2.pointPolygonTest(COURT, (float((x1 + x2) / 2), float(y2)), False) >= 0:
                cv2.fillPoly(m, [poly.astype(np.int32)], 1.0)
    if erode:
        m = cv2.erode(m, np.ones((erode, erode), np.uint8))
    if feather:
        m = cv2.GaussianBlur(m, (0, 0), feather)
    return m[..., None]


def bring_to_front(overlay, frame, matte=None):
    """Re-paste the original player pixels on top of an overlay (players in front)."""
    if matte is None:
        matte = player_matte(frame)
    return (overlay.astype(np.float32) * (1 - matte) + frame.astype(np.float32) * matte).astype(np.uint8)


plt.figure()
_demo = ground_rings(draw_court(frame.copy(), alpha=0.45, thickness=2), tracked)
show(bring_to_front(_demo, frame), "Players composited in front of the ground rings")
plt.show()

### Compose the full overlay

With the hero ring in place, we add the supporting cast — a **motion trail** (where each player has
been), a small **pointer triangle**, the **P1…P4** label, and a translucent **HUD** — all keyed to
the same per-player colour. `render()` stacks them in the right order for every frame.

In [ ]:
triangle = sv.TriangleAnnotator(color=PALETTE, base=22, height=18, color_lookup=LK)
trace    = sv.TraceAnnotator(color=PALETTE, thickness=3, trace_length=FPS,
                             position=sv.Position.BOTTOM_CENTER, color_lookup=LK)
label    = sv.LabelAnnotator(color=PALETTE, text_color=sv.Color.BLACK, text_scale=0.55,
                             text_position=sv.Position.BOTTOM_CENTER, color_lookup=LK)


def hud(img, n_players, t_sec):
    '''Translucent broadcast-style info panel, top-left.'''
    x, y, w, h = 24, 24, 380, 104
    ov = img.copy()
    cv2.rectangle(ov, (x, y), (x + w, y + h), (18, 18, 18), -1)
    img = cv2.addWeighted(ov, 0.55, img, 0.45, 0)
    cv2.rectangle(img, (x, y), (x + w, y + h), (255, 255, 255), 1, cv2.LINE_AA)
    cv2.putText(img, "PADEL ANALYTICS", (x + 18, y + 38),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(img, f"players tracked: {n_players}", (x + 18, y + 70),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2, cv2.LINE_AA)
    cv2.putText(img, f"t = {t_sec:5.2f}s", (x + 18, y + 94),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1, cv2.LINE_AA)
    return img


def render(frame, dets, t_sec=0.0, trail=False):
    out = draw_court(frame.copy(), alpha=0.45, thickness=2)
    if trail:
        out = trace.annotate(out, dets)
    out = ground_rings(out, dets)                 # the AR hero ring
    out = triangle.annotate(out, dets)
    # if dets.tracker_id is not None and len(dets):
    #     out = label.annotate(out, dets, [f"P{int(t)}" for t in dets.tracker_id])
    if FOREGROUND:                                # players sit in front of the ground ring
        out = bring_to_front(out, frame)
    return hud(out, len(dets), t_sec)


plt.figure()
show(render(frame, tracked, t_sec=300 / FPS, trail=False), "The full AR overlay on a single frame")
plt.show()

## 6. Render the hero reel

Now we run the full loop — **detect → filter to court → track → render** — over a short slice of the
match and write it to video. We also collect each player's foot position per frame for the analytics
in the next section.

Pick any window with `START_SEC` / `DURATION_SEC`. A short, punchy clip makes the best LinkedIn/GIF.

In [ ]:
# --- Stabilizer: EMA-smooth each track + keep it alive briefly when it's lost ---
class TrackStabilizer:
    """Smooths boxes (less jitter) and re-emits a track at its last position
    for up to `hold_frames` after the tracker drops it (no blinking rings)."""

    def __init__(self, smoothing=0.35, hold_frames=30):
        self.alpha = float(smoothing)      # EMA weight on the new box (lower = steadier, more lag)
        self.hold = int(hold_frames)       # frames to keep coasting a lost track (~0.5s @ 60fps)
        self._state = {}                   # tracker_id -> {xyxy, conf, last_seen}

    def update(self, dets, frame_idx):
        if dets.tracker_id is not None:
            confs = dets.confidence if dets.confidence is not None else np.ones(len(dets))
            for xyxy, tid, conf in zip(dets.xyxy, dets.tracker_id, confs):
                tid = int(tid); xyxy = np.asarray(xyxy, dtype=float)
                st = self._state.get(tid)
                st = {"xyxy": xyxy.copy()} if st is None else \
                     {"xyxy": self.alpha * xyxy + (1 - self.alpha) * st["xyxy"]}
                st["conf"] = float(conf); st["last_seen"] = frame_idx
                self._state[tid] = st

        ids, boxes, confs = [], [], []
        for tid, st in list(self._state.items()):
            if frame_idx - st["last_seen"] > self.hold:     # expired -> forget
                del self._state[tid]; continue
            ids.append(tid); boxes.append(st["xyxy"]); confs.append(st["conf"])
        if not ids:
            return sv.Detections.empty()
        return sv.Detections(
            xyxy=np.array(boxes, dtype=float),
            confidence=np.array(confs, dtype=float),
            class_id=np.zeros(len(ids), dtype=int),
            tracker_id=np.array(ids, dtype=int),
        )


START_SEC, DURATION_SEC = 0.0, 2.0          # which slice of the match to render
SMOOTHING, HOLD_FRAMES = 0.35, 30           # stabilization knobs (see TrackStabilizer)
start = int(START_SEC * FPS)
n_frames = int(DURATION_SEC * FPS)

reel = OUT / "padel_ar_reel.mp4"
out_info = sv.VideoInfo(width=info.width, height=info.height, fps=FPS)

tracker = new_tracker()
stabilizer = TrackStabilizer(SMOOTHING, HOLD_FRAMES)
tracks = []                                  # (frame, id, x_foot, y_foot)

cap = cv2.VideoCapture(str(VIDEO)); cap.set(cv2.CAP_PROP_POS_FRAMES, start)
with sv.VideoSink(str(reel), out_info) as sink:
    for i in tqdm(range(n_frames), desc="Rendering reel"):
        ok, f = cap.read()
        if not ok:
            break
        d = detector.detect(f)
        d = d[zone.trigger(d)]
        d = tracker.update_with_detections(d)
        d = stabilizer.update(d, i)          # <-- smooth + coast through lost frames
        sink.write_frame(render(f, d, t_sec=START_SEC + i / FPS))
        if d.tracker_id is not None:
            for xyxy, tid in zip(d.xyxy, d.tracker_id):
                tracks.append((i, int(tid), float((xyxy[0] + xyxy[2]) / 2), float(xyxy[3])))
cap.release()

playable = to_h264(reel, OUT / "padel_ar_reel_h264.mp4")
print(f"Reel: {playable}  ({len(tracks)} player-detections collected)")


### ▶️ The reel

A compact, looping GIF is embedded below so the notebook is self-contained; the full-resolution MP4 is saved alongside it in `data/processed/`.

In [ ]:
import subprocess
from IPython.display import Image

gif = OUT / "padel_ar_demo.gif"
# Downscaled + lower-fps GIF keeps the embedded preview light (and shareable).
subprocess.run(
    ["ffmpeg", "-y", "-i", str(playable),
     "-vf", "fps=12,scale=600:-1:flags=lanczos", "-loop", "0", str(gif)],
    check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
print(f"Full-res MP4 : {playable}")
print(f"Preview GIF  : {gif}  ({gif.stat().st_size/1e6:.1f} MB)")
Image(filename=str(gif))

## 7. From tracks to analytics — player movement

The tracking we just did isn't only pretty — it's *data*. Each player's foot position over time is a
trajectory. Plotting all four on a single court view gives an immediate read on **court coverage and
positioning** — the foundation for the heatmaps, distance-covered and shot-zone stats coming in later
stages.

In [ ]:
import pandas as pd

df = pd.DataFrame(tracks, columns=["frame", "player", "x", "y"])
# Keep the four most-tracked IDs (the on-court players) for a clean plot.
top = df["player"].value_counts().nlargest(4).index.tolist()

bg = draw_court(grab(start).copy(), alpha=0.35, thickness=2)
plt.figure(figsize=(13, 7))
plt.imshow(cv2.cvtColor(bg, cv2.COLOR_BGR2RGB))
hexes = ["#00E5FF", "#FF3DAE", "#FFD23F", "#7CFF6B"]
for color, pid in zip(hexes, top):
    g = df[df["player"] == pid]
    plt.plot(g["x"], g["y"], "-", color=color, lw=2.5, alpha=0.9, label=f"P{pid}")
    plt.scatter(g["x"].iloc[-1], g["y"].iloc[-1], color=color, s=80, edgecolor="white", zorder=5)
plt.axis("off"); plt.legend(loc="upper right", framealpha=0.6)
plt.title("Player movement trajectories over the reel", fontsize=13, weight="bold")
plt.show()

# Rough distance covered (in pixels) per player — a teaser for real-world metres after homography.
dist = (df.sort_values("frame")
          .groupby("player")[["x", "y"]]
          .apply(lambda p: float(np.hypot(*np.diff(p.values, axis=0).T).sum())))
print("Approx. pixel distance covered:")
print(dist.loc[top].round(0).to_string())

## 8. Player movement heatmap

Where did each player spend their time? We split the court into a **grid of zones** and shade each one
by how long a player stood in it — green (passed through) → red (camped there) — exactly like the
zonal heatmaps on a football broadcast.

The grid lives in the **court's own plane**: we map the court quad to a unit square with a homography,
tile that square into cells, then project each cell back — so the rectangles follow the court's
perspective (smaller at the far end). We desaturate the court underneath so the colours read true over
the blue/orange surface.

A single rally barely moves, so we first run a quick detection pass over a **longer window**.

### Step 1 — mark the court corners

The grid is only as good as the court outline behind it. Drag the four corners
(**TL, TR, BR, BL** = top-left, top-right, bottom-right, bottom-left of the playing surface) until the
preview grid sits flat on the court. Your choice is saved to `COURT_QUAD` and drives the homography for
every map below — re-run this for a new clip and everything downstream follows.

In [ ]:
import ipywidgets as widgets

UNIT = np.float32([(0, 0), (1, 0), (1, 1), (0, 1)])           # TL, TR, BR, BL of a unit square
COURT_QUAD = np.float32([(715, 240), (1250, 240),            # default = your saved corners
                         (1530, 1070), (455, 1065)])          # TL, TR, BR, BL
_pick_bg = grab(300)
_NAMES = ["TL", "TR", "BR", "BL"]


def court_grid_preview(img, quad, nx=12, ny=8):
    """Draw the perspective grid + labelled corners for a given court quad."""
    Hc = cv2.getPerspectiveTransform(UNIT, quad.astype(np.float32))
    for i in range(nx + 1):
        ln = cv2.perspectiveTransform(np.float32([(i / nx, 0), (i / nx, 1)]).reshape(-1, 1, 2), Hc)
        cv2.polylines(img, [ln.reshape(-1, 2).astype(np.int32)], False, (255, 255, 255), 1, cv2.LINE_AA)
    for j in range(ny + 1):
        ln = cv2.perspectiveTransform(np.float32([(0, j / ny), (1, j / ny)]).reshape(-1, 1, 2), Hc)
        cv2.polylines(img, [ln.reshape(-1, 2).astype(np.int32)], False, (255, 255, 255), 1, cv2.LINE_AA)
    cv2.polylines(img, [quad.astype(np.int32)], True, (0, 255, 255), 2, cv2.LINE_AA)
    for (px, py), name in zip(quad.astype(int), _NAMES):
        cv2.circle(img, (int(px), int(py)), 8, (0, 0, 255), -1, cv2.LINE_AA)
        cv2.putText(img, name, (int(px) + 12, int(py) - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2, cv2.LINE_AA)
    return img


def _pick(**kw):
    global COURT_QUAD
    COURT_QUAD = np.float32([[kw["TLx"], kw["TLy"]], [kw["TRx"], kw["TRy"]],
                            [kw["BRx"], kw["BRy"]], [kw["BLx"], kw["BLy"]]])
    plt.figure(figsize=(12, 7))
    show(court_grid_preview(_pick_bg.copy(), COURT_QUAD), "Align the grid to the court — drag the 4 corners")
    plt.show()


_sliders = {}
for _name, (_x, _y) in zip(_NAMES, COURT_QUAD):
    _sliders[f"{_name}x"] = widgets.IntSlider(value=int(_x), min=0, max=info.width, step=2,
                                              description=f"{_name} x", continuous_update=False)
    _sliders[f"{_name}y"] = widgets.IntSlider(value=int(_y), min=0, max=info.height, step=2,
                                              description=f"{_name} y", continuous_update=False)
widgets.interact(_pick, **_sliders);


In [ ]:
import pandas as pd


def collect_positions(start_sec=0.0, dur_sec=24.0, stride=3):
    """Detect -> court filter -> track over a window; return DataFrame(frame, player, x, y)."""
    tr, rows = new_tracker(), []
    s, n = int(start_sec * FPS), int(dur_sec * FPS)
    cap = cv2.VideoCapture(str(VIDEO)); cap.set(cv2.CAP_PROP_POS_FRAMES, s)
    for k in tqdm(range(n), desc="Heatmap detection pass"):
        ok, f = cap.read()
        if not ok:
            break
        if k % stride:                                  # sample every `stride`-th frame
            continue
        d = detector.detect(f); d = d[zone.trigger(d)]; d = tr.update_with_detections(d)
        if d.tracker_id is not None:
            for xyxy, tid in zip(d.xyxy, d.tracker_id):
                rows.append((k, int(tid), float((xyxy[0] + xyxy[2]) / 2), float(xyxy[3])))
    cap.release()
    return pd.DataFrame(rows, columns=["frame", "player", "x", "y"])


# Court corners come from Step 1's picker; fall back to COURT if it wasn't run.
UNIT = np.float32([(0, 0), (1, 0), (1, 1), (0, 1)])
COURT_QUAD = globals().get("COURT_QUAD", COURT[[0, 1, 3, 4]].astype(np.float32))


def _heat_lut(name="RdYlGn_r"):
    """256-entry BGR lookup table for a matplotlib colormap (green=low -> red=high)."""
    return (plt.get_cmap(name)(np.linspace(0, 1, 256))[:, :3] * 255).astype(np.uint8)[:, ::-1].copy()


def zonal_heatmap(bg, pts, nx=12, ny=8, alpha=0.68, smooth=0.8, power=0.6,
                  cmap="RdYlGn_r", grid_lines=True):
    """Split the court (COURT_QUAD) into nx*ny perspective zones, shaded by occupancy."""
    quad = COURT_QUAD.astype(np.float32)
    Hc = cv2.getPerspectiveTransform(UNIT, quad)     # unit grid -> image
    Hi = cv2.getPerspectiveTransform(quad, UNIT)     # image -> unit grid
    H_, W_ = bg.shape[:2]

    def cell(i, j):
        u0, u1, v0, v1 = i / nx, (i + 1) / nx, j / ny, (j + 1) / ny
        c = np.float32([(u0, v0), (u1, v0), (u1, v1), (u0, v1)]).reshape(-1, 1, 2)
        return cv2.perspectiveTransform(c, Hc).reshape(-1, 2).astype(np.int32)

    uv = cv2.perspectiveTransform(pts.reshape(-1, 1, 2).astype(np.float32), Hi).reshape(-1, 2)
    inb = (uv[:, 0] >= 0) & (uv[:, 0] < 1) & (uv[:, 1] >= 0) & (uv[:, 1] < 1)
    ci = np.clip((uv[inb, 0] * nx).astype(int), 0, nx - 1)
    cj = np.clip((uv[inb, 1] * ny).astype(int), 0, ny - 1)
    grid = np.zeros((ny, nx), np.float32)
    np.add.at(grid, (cj, ci), 1.0)
    if smooth:
        grid = cv2.GaussianBlur(grid, (0, 0), smooth)
    if grid.max() > 0:
        grid /= grid.max()
    grid = np.power(grid, power)
    lut = _heat_lut(cmap)

    qmask = np.zeros((H_, W_), np.float32)
    cv2.fillConvexPoly(qmask, quad.astype(np.int32), 1.0)
    heat = np.zeros_like(bg)
    for j in range(ny):
        for i in range(nx):
            color = tuple(int(c) for c in lut[int(np.clip(grid[j, i] * 255, 0, 255))])
            cv2.fillConvexPoly(heat, cell(i, j), color, cv2.LINE_AA)

    # Desaturate the court first so heat hues read true over the colourful surface.
    gray3 = cv2.cvtColor(cv2.cvtColor(bg, cv2.COLOR_BGR2GRAY), cv2.COLOR_GRAY2BGR).astype(np.float32)
    qm = qmask[..., None]
    base = bg.astype(np.float32) * (1 - qm) + gray3 * qm
    A = alpha * qm
    out = (base * (1 - A) + heat.astype(np.float32) * A).astype(np.uint8)

    if grid_lines:                                       # faint cell borders
        lines = out.copy()
        for i in range(nx + 1):
            ln = cv2.perspectiveTransform(np.float32([(i / nx, 0), (i / nx, 1)]).reshape(-1, 1, 2), Hc)
            cv2.polylines(lines, [ln.reshape(-1, 2).astype(np.int32)], False, (255, 255, 255), 1, cv2.LINE_AA)
        for j in range(ny + 1):
            ln = cv2.perspectiveTransform(np.float32([(0, j / ny), (1, j / ny)]).reshape(-1, 1, 2), Hc)
            cv2.polylines(lines, [ln.reshape(-1, 2).astype(np.int32)], False, (255, 255, 255), 1, cv2.LINE_AA)
        out = cv2.addWeighted(lines, 0.30, out, 0.70, 0)
    return out


# Quick pass over a longer window, then a combined court-coverage heatmap (all players).
hdf = collect_positions(start_sec=0.0, dur_sec=24.0, stride=3)
htop = hdf["player"].value_counts().nlargest(4).index.tolist()
hbg = grab(300)
hmatte = player_matte(hbg)                       # one segmentation pass; reused per-player below

plt.figure(figsize=(13, 7))
show(bring_to_front(zonal_heatmap(hbg.copy(), hdf[["x", "y"]].values), hbg, hmatte),
     "Court coverage — all four players")
plt.show()

### Per-player heatmaps + distance covered

The same map, one player at a time, each with a broadcast info chip showing how far they travelled.
*(Distance is in pixels for now — it becomes real metres once Stage 3 adds court homography.)*

In [ ]:
def path_length(g):
    """Total path length (pixels) of a player's foot point over the window."""
    p = g.sort_values("frame")[["x", "y"]].values
    return float(np.hypot(*np.diff(p, axis=0).T).sum()) if len(p) > 1 else 0.0


def heat_chip(img, pid, dist):
    """Broadcast-style player + distance chip (top-left of a tile)."""
    color = PALETTE.by_idx(int(pid)).as_bgr()
    x, y, w, h = 20, 20, 300, 84
    ov = img.copy(); cv2.rectangle(ov, (x, y), (x + w, y + h), (18, 18, 18), -1)
    img = cv2.addWeighted(ov, 0.6, img, 0.4, 0)
    cv2.rectangle(img, (x, y), (x + 8, y + h), color, -1)                       # player colour accent
    cv2.putText(img, f"PLAYER {pid}", (x + 22, y + 34),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(img, f"distance  {dist:,.0f} px", (x + 22, y + 66),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)
    return img


fig, axes = plt.subplots(2, 2, figsize=(15, 9))
for ax, pid in zip(axes.ravel(), htop):
    g = hdf[hdf["player"] == pid]
    tile = bring_to_front(zonal_heatmap(hbg.copy(), g[["x", "y"]].values), hbg, hmatte)
    show(heat_chip(tile, pid, path_length(g)), ax=ax)
plt.tight_layout(); plt.show()

## 9. The final cut — full-match film

The full clip with **everything on at once**:

- the **zonal heatmap grid** is live from the very first frame, filling in green → red as play unfolds,
- the **AR ground rings** stay overlaid *on top of* it the whole time (both shown together),
- and the **players are composited in front** of both.

For speed, the heavy models (RF-DETR detector + the segmentation matte) run only every `MODEL_EVERY`
frames and their results are reused in between. Tune `MODEL_EVERY` (higher = faster), `STRIDE` (output
fps) and the grid size at the top of the cell. Uses the court homography from your `court_corners.txt`.

In [ ]:
from collections import defaultdict

# ---------------- controllable settings ----------------
MODEL_EVERY   = 1        # run detector + segmentation matte every Nth processed frame (higher = faster)
STRIDE        = 1        # process/write every Nth source frame (output fps = FPS / STRIDE)
NX, NY        = 12, 8    # heatmap grid resolution
POWER         = 0.4      # heatmap contrast (lower = more spread)
GRID_ALPHA    = 0.62     # heatmap opacity over the court
HEATMAP_AT    = 9.0     # seconds: the heatmap appears at/after this time
HIDE_RINGS_AT = 9.0     # seconds: the ground rings fade out at/after this time
FADE          = 1.0      # seconds: transition length for the heatmap-in / rings-out
MAX_SEC       = None        # None = whole clip; set e.g. 25 to preview faster (must exceed HEATMAP_AT to see it)


# ---------------- court quad from your saved corners ----------------
def _load_corners(path="court_corners.txt"):
    d = {}
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#"):
            n, x, y = line.split(); d[n] = (float(x), float(y))
    return np.float32([d["TL"], d["TR"], d["BR"], d["BL"]])

QUAD9 = _load_corners() if Path("court_corners.txt").exists() else COURT_QUAD.astype(np.float32)
Hh, Ww = info.height, info.width
_U = np.float32([(0, 0), (1, 0), (1, 1), (0, 1)])
_Hc = cv2.getPerspectiveTransform(_U, QUAD9)
_Hi = cv2.getPerspectiveTransform(QUAD9, _U)
_CELLS = [[cv2.perspectiveTransform(
            np.float32([(i/NX, j/NY), ((i+1)/NX, j/NY), ((i+1)/NX, (j+1)/NY), (i/NX, (j+1)/NY)]).reshape(-1, 1, 2),
            _Hc).reshape(-1, 2).astype(np.int32) for i in range(NX)] for j in range(NY)]
_LUT = (plt.get_cmap("RdYlGn_r")(np.linspace(0, 1, 256))[:, :3] * 255).astype(np.uint8)[:, ::-1].copy()

# static layers (built once)
_QMASK = np.zeros((Hh, Ww), np.float32); cv2.fillConvexPoly(_QMASK, QUAD9.astype(np.int32), 1.0)
_GRIDLINES = np.zeros((Hh, Ww, 3), np.uint8)
for _row in _CELLS:
    for _cp in _row:
        cv2.polylines(_GRIDLINES, [_cp], True, (255, 255, 255), 1, cv2.LINE_AA)


def _smooth(x):
    x = float(np.clip(x, 0, 1)); return x * x * (3 - 2 * x)


def _bake_grid(val):
    """Colour every zone by its value (recomputed only when the heatmap changes)."""
    valS = cv2.GaussianBlur(val, (0, 0), 0.8)
    heat = np.zeros((Hh, Ww, 3), np.uint8)
    for j in range(NY):
        for i in range(NX):
            cv2.fillConvexPoly(heat, _CELLS[j][i], tuple(int(x) for x in _LUT[int(np.clip(valS[j, i] * 255, 0, 255))]), cv2.LINE_AA)
    return heat


def _compose_grid(bg, heat, alpha):
    """Desaturate the court + overlay the coloured grid, scaled by `alpha` (0 = hidden)."""
    if alpha <= 0:
        return bg
    gray3 = cv2.cvtColor(cv2.cvtColor(bg, cv2.COLOR_BGR2GRAY), cv2.COLOR_GRAY2BGR).astype(np.float32)
    A = (alpha * _QMASK)[..., None]
    base = bg.astype(np.float32) * (1 - A) + gray3 * A
    Ah = (GRID_ALPHA * alpha * _QMASK)[..., None]
    out = (base * (1 - Ah) + heat.astype(np.float32) * Ah).astype(np.uint8)
    return cv2.addWeighted(out, 1.0, _GRIDLINES, 0.22 * alpha, 0)


def _final_hud(img, t, phase):
    x, y, w, h = 24, 24, 360, 70
    ov = img.copy(); cv2.rectangle(ov, (x, y), (x + w, y + h), (18, 18, 18), -1)
    img = cv2.addWeighted(ov, 0.55, img, 0.45, 0)
    cv2.rectangle(img, (x, y), (x + w, y + h), (255, 255, 255), 1, cv2.LINE_AA)
    cv2.putText(img, "PADEL ANALYTICS", (x + 16, y + 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(img, f"t={t:5.1f}s   {phase}", (x + 16, y + 56), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1, cv2.LINE_AA)
    return img


# ---------------- render the film ----------------
final = OUT / "padel_final.mp4"
tracker = new_tracker(); stab = TrackStabilizer()
grids = defaultdict(lambda: np.zeros((NY, NX), np.float32))
last_d, last_matte, heat = None, None, np.zeros((Hh, Ww, 3), np.uint8)
total = info.total_frames if MAX_SEC is None else int(MAX_SEC * FPS)
out_info = sv.VideoInfo(width=Ww, height=Hh, fps=max(1, FPS // STRIDE))

cap = cv2.VideoCapture(str(VIDEO)); proc = 0
with sv.VideoSink(str(final), out_info) as sink:
    for k in tqdm(range(total), desc="Final cut"):
        ok, f = cap.read()
        if not ok:
            break
        if k % STRIDE:
            continue
        if proc % MODEL_EVERY == 0:                          # run the heavy models, refresh the heatmap
            d = detector.detect(f); d = d[zone.trigger(d)]
            d = tracker.update_with_detections(d); d = stab.update(d, k)
            last_d, last_matte = d, player_matte(f)
            if d.tracker_id is not None:                     # accumulate occupancy (live, from frame 0)
                for xyxy, tid in zip(d.xyxy, d.tracker_id):
                    uv = cv2.perspectiveTransform(np.float32([[(xyxy[0] + xyxy[2]) / 2, xyxy[3]]]).reshape(-1, 1, 2), _Hi).reshape(-1)
                    if 0 <= uv[0] < 1 and 0 <= uv[1] < 1:
                        grids[int(tid)][int(np.clip(uv[1] * NY, 0, NY - 1)), int(np.clip(uv[0] * NX, 0, NX - 1))] += 1
            tot = np.sum(list(grids.values()), axis=0) if grids else np.zeros((NY, NX), np.float32)
            heat = _bake_grid(np.power(tot / max(tot.max(), 1.0), POWER))
        proc += 1

        t = k / FPS
        heat_alpha = _smooth((t - HEATMAP_AT) / FADE)        # 0 before HEATMAP_AT -> 1 after
        ring_alpha = 1.0 - _smooth((t - HIDE_RINGS_AT) / FADE)  # 1 before HIDE_RINGS_AT -> 0 after

        out = _compose_grid(f, heat, heat_alpha)             # heatmap appears at HEATMAP_AT
        if ring_alpha > 0 and last_d is not None:            # rings hide at HIDE_RINGS_AT
            rl = ground_rings(out.copy(), last_d)
            out = cv2.addWeighted(rl, ring_alpha, out, 1.0 - ring_alpha, 0)
        if last_matte is not None:
            out = bring_to_front(out, f, last_matte)         # players in front of everything
        out = _final_hud(out, t, "LIVE TRACKING" if t < HEATMAP_AT else "LIVE HEATMAP")
        sink.write_frame(out)
cap.release()

playable = to_h264(final, OUT / "padel_final_h264.mp4")
print("Final cut ->", playable)


In [ ]:
import subprocess
from IPython.display import Image

# Looping preview of the opening (grid live from the start, rings overlaid).
gif = OUT / "padel_final_preview.gif"
subprocess.run(
    ["ffmpeg", "-y", "-t", "14", "-i", str(playable),
     "-vf", "fps=12,scale=640:-1:flags=lanczos", "-loop", "0", str(gif)],
    check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
print("Preview:", gif)
Image(filename=str(gif))

## 🔭 What's next

This notebook delivers Stages 1–2 of the pipeline with a broadcast-quality overlay. The roadmap from
here:

- **Stage 3 — Court detection & homography** → replace the hand-drawn polygon, unlock a true top-down
  minimap and convert pixel distances into **metres**.
- **Stage 4 — Movement analytics** → heatmaps, distance covered, average court position per player.
- **Stage 5 — Ball detection** → a dedicated small-object model for trajectory & speed.
- **Stage 6 — Shot/event detection** → bandeja, víbora, smash, volley.

---

*Built by **Nassim** — computer-vision & MLOps. RF-DETR · ByteTrack · Supervision.*
